============================================================================

# Pipeline de ingesta ómico para MVP-2: Encoders Ómicos
 
 
**Autor:** Juan Carlos Barajas Alarcón

**Proyecto:** Inferencia del envejecimiento celular (Tesis Maestría, Anáhuac)

**Fecha:** Marzo 2026

============================================================================
 
**SECCIONES:**

  0. CONFIG — Rutas a ajustar según tu máquina
  1. IMPORTS
  2. CARGAR CSV CENTRAL — Lifespan_Study_Data.csv (1919×271)




**Objetivo:** Construir un manifest *sample-centric* que alimente los encoders de RNA-seq,
metilación y biomarcadores escalares, compatibles con el contrato de interfaz de MVP-1
(z ∈ R^256) para fusión multimodal via concat+mask. El z latente fusionado debe capturar el estado global de la célula,
del cual predeciremos Hallmarks of Aging e Índice Continuo de Daño (ICD).

**Diferencia con manifest MVP-1:** MVP-1 es *image-centric* (una fila por imagen, 2433 filas).
MVP-2 es *sample-centric* (una fila por sample_id del CSV central, ~1919 filas).
Cada fila lleva flags de qué modalidades están disponibles y las llaves para acceder
a cada matriz ómica.

**Inputs verificados (exploración celda por celda):**
- CSV central: `Lifespan_Study_Data.csv` (1919 × 271), llave = `Sample`
- RNA processed: 28,633 genes × 345 muestras (VST), cols = `Sample_N`
- Met processed: ~866K CpGs × 479 muestras (betas), cols = `{sentrix} beta`
- GEO metadata RNA/Met: puentes de join verificados con 0 pérdidas
- Manifest MVP-1: folds por cell_line, embeddings A2

**Outputs:**
- `manifest_mvp2_master_{ts}.parquet` — tabla maestra sample-centric
- `mvp2_rna_selected_{ts}.parquet` — submatriz RNA (muestras × HVGs)
- `mvp2_met_selected_{ts}.parquet` — submatriz Met (muestras × top CpGs)
- `mvp2_selected_genes_{ts}.json` — lista de genes + hallmark mapping
- Subconjuntos: pretrain / finetune / eval

## 0. CONFIG — Ajustar rutas

In [29]:
from pathlib import Path

CONFIG = {
    # --- Datos de Sturm ---
    'csv_central': Path('/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv'),
    'geo_rna_meta': Path('/Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv'),
    'geo_met_meta': Path('/Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv'),
    'rna_processed': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179848/GSE179848_processed_cell_lifespan_RNAseq_data.csv.gz'),
    'rna_raw_counts': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179848/GSE179848_raw_counts_cell_lifespan_RNAseq_data.csv.gz'),
    'met_processed': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179847/GSE179847_Cell_lifespan_DNAm_processed_matrix.csv.gz'),
    'manifest_mvp1': Path('/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_full_20260327_152829.csv'),
    'mvp1_embeddings': Path('/Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final/A2_final_embeddings.csv'),
    
    # Output
    'output_dir': Path('/Users/JCB/Documentos/Proyecto Integrador/data/manifests/'),

    # --- Parámetros ---
    'n_hvgs':         2000,     # Top genes más variables para encoder RNA
    'n_cpgs_var':     10000,    # Top CpGs más variables para encoder Met
    'latent_dim':     256,      # Contrato de interfaz con MVP-1
}

for name, path in CONFIG.items():
    if isinstance(path, Path):
        status = "✅" if path.exists() else "❌"
        print(f"  {status} {name}: {path.name}")

  ✅ csv_central: Lifespan_Study_Data.csv
  ✅ geo_rna_meta: GSE179848_sample_metadata.csv
  ✅ geo_met_meta: GSE179847_sample_metadata.csv
  ✅ rna_processed: GSE179848_processed_cell_lifespan_RNAseq_data.csv.gz
  ✅ rna_raw_counts: GSE179848_raw_counts_cell_lifespan_RNAseq_data.csv.gz
  ✅ met_processed: GSE179847_Cell_lifespan_DNAm_processed_matrix.csv.gz
  ✅ manifest_mvp1: manifest_full_20260327_152829.csv
  ✅ mvp1_embeddings: A2_final_embeddings.csv
  ✅ output_dir: manifests


## 1. Imports

In [30]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import hashlib
import json
import gzip
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f"Timestamp: {TIMESTAMP}")

Timestamp: 20260328_143235


## 2. Cargar CSV central y construir tabla maestra

La tabla maestra tiene **una fila por `Sample`** del CSV central (1919 filas).
Renombramos columnas para consistencia con el manifest MVP-1.

In [31]:
df_raw = pd.read_csv(CONFIG['csv_central'], low_memory=False)
print(f"CSV central: {df_raw.shape}")

# ── Renombrar columnas (mismo mapeo que manifest MVP-1) ───────────
COL_MAP = {
    'Sample':                   'sample_id',
    'Cell_Line':                'cell_line',
    'Passage':                  'passage',
    'Study_Part':               'study_part',
    'Percent_Oxygen':           'percent_oxygen',
    'Treatments':               'treatment_clean',
    'Treatment':                'treatment_full',
    'Population_Doublings_DI':  'pdl',
    'Percent_Lifespan':         'percent_lifespan',
    'Clinical_Condition':       'clinical_condition',
    'Donor_Age':                'donor_age',
    'Cell_Type':                'cell_type',
    'Sex':                      'sex',
    'Days_Grown':               'days_grown',
    'Telomere_Length':           'telomere_length',
    'Telomere_Length_CV':        'telomere_length_cv',
    'Copy_Number':              'mtdna_cn',
    'RNAseq_ID':                'rnaseq_id',
    'basename':                 'methylation_basename',
    # Relojes epigenéticos
    'Horvath1':                 'clock_horvath1',
    'Horvath2':                 'clock_horvath2',
    'Hannum':                   'clock_hannum',
    'PhenoAge':                 'clock_phenoage',
    'GrimAge':                  'clock_grimage',
    'DNAmTL':                   'clock_dnamtl',
    'DNAmSen':                  'clock_dnamsen',
    'Mol_Skin_Clock':           'clock_skin',
    'DunedinPoAm_38':           'clock_dunedin',
    # Componentes GrimAge
    'DNAmADM':                  'grim_adm',
    'DNAmB2M':                  'grim_b2m',
    'DNAmCystatinC':            'grim_cystatin',
    'DNAmGDF15':                'grim_gdf15',
    'DNAmLeptin':               'grim_leptin',
    'DNAmPAI1':                 'grim_pai1',
    'DNAmTIMP1':                'grim_timp1',
}

existing = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
missing = {k: v for k, v in COL_MAP.items() if k not in df_raw.columns}
if missing:
    print(f"⚠️ Columnas no encontradas: {list(missing.keys())}")

master = df_raw[list(existing.keys())].rename(columns=existing).copy()
master['sample_id'] = master['sample_id'].astype(int)
master['passage'] = pd.to_numeric(master['passage'], errors='coerce')
master['pdl'] = pd.to_numeric(master['pdl'], errors='coerce')

assert master['sample_id'].is_unique, "sample_id NO es único"
print(f"✅ Tabla maestra: {master.shape}, sample_id único")
print(f"   Cell lines: {sorted(master['cell_line'].dropna().unique())}")

CSV central: (1919, 271)
✅ Tabla maestra: (1919, 35), sample_id único
   Cell lines: ['HEK293', 'hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB6', 'hFB7', 'hFB8']


## 3. PDL normalizado por cell_line + bins

In [32]:
# PDL_norm ∈ [0,1] — 0=más joven de esa línea, 1=más viejo
# Mismo cálculo que manifest MVP-1
master['pdl_norm'] = np.nan
for cl in master['cell_line'].dropna().unique():
    mask = master['cell_line'] == cl
    pdl_vals = master.loc[mask, 'pdl']
    if pdl_vals.notna().sum() > 1:
        pmin, pmax = pdl_vals.min(), pdl_vals.max()
        if pmax > pmin:
            master.loc[mask, 'pdl_norm'] = (pdl_vals - pmin) / (pmax - pmin)
        else:
            master.loc[mask, 'pdl_norm'] = 0.5

master['pdl_bin'] = pd.cut(
    master['pdl_norm'],
    bins=[-0.01, 0.33, 0.66, 1.01],
    labels=['early', 'mid', 'late']
)

print(f"PDL_norm: {master['pdl_norm'].notna().sum()} valores, rango [{master['pdl_norm'].min():.3f}, {master['pdl_norm'].max():.3f}]")
print(f"PDL bins: {master['pdl_bin'].value_counts().to_dict()}")

PDL_norm: 1900 valores, rango [0.000, 1.000]
PDL bins: {'late': 689, 'mid': 617, 'early': 594}


## 4. Construir lookups GEO (verificados en exploración)

```
RNA:  Sample_N (col matriz) → N=char_rnaseq_sampleid → char_study_sample_id = sample_id
MET:  sentrix_basename (col matriz) → char_slide+"_"+char_array → char_study_sample_id = sample_id
```

In [33]:
geo_rna = pd.read_csv(CONFIG['geo_rna_meta'])
geo_met = pd.read_csv(CONFIG['geo_met_meta'])

# ── RNA lookup: rnaseq_sampleid → sample_id del CSV central ──────
rna_lookup_to_sample = dict(zip(
    geo_rna['char_rnaseq_sampleid'].astype(int),
    geo_rna['char_study_sample_id'].astype(int)
))
print(f"RNA lookup: {len(rna_lookup_to_sample)} entradas (rnaseq_id → sample_id)")

# ── MET lookup: sentrix_basename → sample_id del CSV central ─────
geo_met = geo_met.copy()
geo_met['sentrix_basename'] = (
    geo_met['char_slide'].astype(str).str.strip()
    + '_'
    + geo_met['char_array'].astype(str).str.strip()
)
met_lookup_to_sample = dict(zip(
    geo_met['sentrix_basename'],
    geo_met['char_study_sample_id'].astype(int)
))
print(f"MET lookup: {len(met_lookup_to_sample)} entradas (sentrix → sample_id)")

# ── Verificar contra master ──────────────────────────────────────
master_ids = set(master['sample_id'])
rna_sample_ids = set(rna_lookup_to_sample.values())
met_sample_ids = set(met_lookup_to_sample.values())

print(f"\nRNA samples en CSV central: {len(rna_sample_ids & master_ids)} / {len(rna_sample_ids)}")
print(f"MET samples en CSV central: {len(met_sample_ids & master_ids)} / {len(met_sample_ids)}")

# Inversos: sample_id → rnaseq_id, sample_id → sentrix_basename
sample_to_rnaseq = {v: k for k, v in rna_lookup_to_sample.items()}
sample_to_sentrix = {v: k for k, v in met_lookup_to_sample.items()}

RNA lookup: 345 entradas (rnaseq_id → sample_id)
MET lookup: 479 entradas (sentrix → sample_id)

RNA samples en CSV central: 345 / 345
MET samples en CSV central: 479 / 479


## 5. Masks de disponibilidad

Estas masks son esenciales para:
1. Saber qué columnas de las matrices ómicas cargar para cada muestra
2. Drop-modality training (simular missingness)
3. Concat+mask fusion

In [45]:
# ── Ómicas (desde lookups GEO) ─────────────────────────────────────
master['has_rna'] = master['sample_id'].isin(rna_sample_ids)
master['has_met'] = master['sample_id'].isin(met_sample_ids)

# ── Biomarcadores escalares (desde CSV central) ───────────────────
master['has_telomere'] = master['telomere_length'].notna()
master['has_mtdna'] = master['mtdna_cn'].notna()

# ── Imagen (desde manifest MVP-1) ────────────────────────────────
manifest_mvp1 = pd.read_csv(CONFIG['manifest_mvp1'], low_memory=False)
img_sample_ids = set(manifest_mvp1['sample_id'].dropna().astype(int).unique())
master['has_img'] = master['sample_id'].isin(img_sample_ids)

# ── Relojes epigenéticos ─────────────────────────────────────────
clock_cols = [c for c in master.columns if c.startswith('clock_')]
master['has_clocks'] = master[clock_cols].notna().any(axis=1)

# ── Conteo de modalidades ────────────────────────────────────────
modal_flags = ['has_rna', 'has_met', 'has_img', 'has_telomere', 'has_mtdna']
master['n_modalities'] = master[modal_flags].sum(axis=1)

# ── Llaves de acceso a matrices ──────────────────────────────────
# Estas columnas indican QUÉ columna leer de cada matriz
master['rna_matrix_col'] = master['sample_id'].map(
    lambda x: f"Sample_{sample_to_rnaseq[x]}" if x in sample_to_rnaseq else None
)
master['met_matrix_col'] = master['sample_id'].map(
    lambda x: sample_to_sentrix[x] if x in sample_to_sentrix else None
)

# ── Resumen ──────────────────────────────────────────────────────
print("DISPONIBILIDAD POR MODALIDAD:")
print(f"  Total muestras:  {len(master)}")
print(f"  Con PDL:         {master['pdl'].notna().sum()}")
print(f"  Con RNA:         {master['has_rna'].sum()}")
print(f"  Con metilación:  {master['has_met'].sum()}")
print(f"  Con telómero:    {master['has_telomere'].sum()}")
print(f"  Con mtDNA CN:    {master['has_mtdna'].sum()}")
print(f"  Con imagen:      {master['has_img'].sum()}")
print(f"  Con relojes:     {master['has_clocks'].sum()}")

print(f"\nPAIRING MULTIMODAL:")
r, m, i = master['has_rna'], master['has_met'], master['has_img']
print(f"  RNA ∩ Met:         {(r & m).sum()}")
print(f"  RNA ∩ Img:         {(r & i).sum()}")
print(f"  Met ∩ Img:         {(m & i).sum()}")
print(f"  RNA ∩ Met ∩ Img:   {(r & m & i).sum()}")
print(f"  Cualquier ómica:   {(r | m).sum()}")
print(f"  Ómica + PDL:       {((r | m) & master['pdl'].notna()).sum()}")

DISPONIBILIDAD POR MODALIDAD:
  Total muestras:  1919
  Con PDL:         1918
  Con RNA:         345
  Con metilación:  479
  Con telómero:    496
  Con mtDNA CN:    493
  Con imagen:      973
  Con relojes:     496

PAIRING MULTIMODAL:
  RNA ∩ Met:         109
  RNA ∩ Img:         148
  Met ∩ Img:         254
  RNA ∩ Met ∩ Img:   32
  Cualquier ómica:   715
  Ómica + PDL:       715


## 6. Herencia de folds de MVP-1

Mismos folds GroupKFold por cell_line para consistencia.

In [35]:
FOLD_ASSIGNMENT = {
    'hFB12': 0,
    'hFB13': 1, 'hFB14': 1,
    'hFB1':  2, 'hFB2':  2, 'hFB11': 2,
    'hFB6':  1,   # SURF1
    'hFB7':  2,   # SURF1
    'hFB8':  1,   # SURF1
}

master['fold'] = master['cell_line'].map(FOLD_ASSIGNMENT)

# Resumen por fold con desglose de modalidades
print("MUESTRAS POR FOLD:")
for f in sorted(master['fold'].dropna().unique()):
    mf = master[master['fold'] == f]
    print(f"  Fold {int(f)}: {len(mf)} total | "
          f"RNA={mf['has_rna'].sum()} | Met={mf['has_met'].sum()} | "
          f"Img={mf['has_img'].sum()} | PDL={mf['pdl'].notna().sum()}")

n_no_fold = master['fold'].isna().sum()
if n_no_fold > 0:
    orphan_lines = master.loc[master['fold'].isna(), 'cell_line'].unique()
    print(f"\n⚠️ {n_no_fold} muestras sin fold (cell_lines: {orphan_lines})")

MUESTRAS POR FOLD:
  Fold 0: 496 total | RNA=111 | Met=143 | Img=184 | PDL=496
  Fold 1: 873 total | RNA=181 | Met=196 | Img=469 | PDL=872
  Fold 2: 477 total | RNA=47 | Met=140 | Img=320 | PDL=477

⚠️ 73 muestras sin fold (cell_lines: <StringArray>
['HEK293', nan]
Length: 2, dtype: str)


## 7. Cargar RNA-seq y seleccionar genes (HVGs + hallmarks)

28,633 genes → ~2,000 HVGs + genes hallmark forzados.

In [36]:
rna = pd.read_csv(CONFIG['rna_processed'], compression='gzip', index_col=0)
print(f"RNA matrix: {rna.shape[0]} genes × {rna.shape[1]} muestras")

# ── Genes hallmark que forzamos en la selección ───────────────────
HALLMARK_GENES = {
    'senescence': [
        'CDKN2A', 'CDKN1A', 'TP53', 'RB1', 'GLB1', 'LMNB1', 'HMGA1', 'HMGA2',
    ],
    'sasp': [
        'IL6', 'CXCL8', 'IL1A', 'IL1B', 'MMP1', 'MMP3', 'MMP9',
        'SERPINE1', 'CCL2', 'IGFBP3', 'IGFBP5', 'IGFBP7',
    ],
    'mitochondrial': [
        'TFAM', 'PPARGC1A', 'SOD2', 'PINK1',  # nucleares (MT-* no están en la matriz)
    ],
    'telomere': ['TERT', 'TERC', 'DKC1', 'POT1'],
    'genomic_instability': ['ATM', 'ATR', 'BRCA1', 'BRCA2', 'CHEK1', 'CHEK2'],
    'nutrient_sensing': ['MTOR', 'SIRT1', 'SIRT3', 'SIRT6', 'IGF1', 'IGF1R'],
}

available_genes = set(rna.index)
hallmark_found = set()
hallmark_missing = set()
for cat, genes in HALLMARK_GENES.items():
    found = [g for g in genes if g in available_genes]
    miss = [g for g in genes if g not in available_genes]
    hallmark_found.update(found)
    hallmark_missing.update(miss)
    status = "✅" if not miss else "⚠️"
    print(f"  {status} {cat}: {len(found)}/{len(genes)}", end="")
    if miss:
        print(f"  (faltantes: {miss})", end="")
    print()

# ── HVGs por varianza ────────────────────────────────────────────
gene_var = rna.var(axis=1).sort_values(ascending=False)
hvg_top = set(gene_var.head(CONFIG['n_hvgs']).index)

# ── Unión: HVGs + hallmarks forzados ─────────────────────────────
selected_genes = sorted(hvg_top | hallmark_found)
overlap = len(hvg_top & hallmark_found)
print(f"\nSelección final:")
print(f"  HVGs top-{CONFIG['n_hvgs']}: {len(hvg_top)}")
print(f"  Hallmark forzados: {len(hallmark_found)}")
print(f"  Overlap: {overlap}")
print(f"  Total genes: {len(selected_genes)}")

# ── Extraer submatriz (genes × muestras → transponer a muestras × genes) ──
rna_selected = rna.loc[rna.index.isin(selected_genes)].T  # → muestras × genes
rna_selected.index.name = 'rna_col'
print(f"\nSubmatriz RNA: {rna_selected.shape} (muestras × genes)")

RNA matrix: 28633 genes × 345 muestras
  ✅ senescence: 8/8
  ✅ sasp: 12/12
  ✅ mitochondrial: 4/4
  ✅ telomere: 4/4
  ✅ genomic_instability: 6/6
  ✅ nutrient_sensing: 6/6

Selección final:
  HVGs top-2000: 2000
  Hallmark forzados: 40
  Overlap: 13
  Total genes: 2027

Submatriz RNA: (345, 2027) (muestras × genes)


## 8. Cargar metilación (solo betas) y seleccionar CpGs por varianza

~866K CpGs → top 10K por varianza. Carga en float32 (~1.5 GB RAM).

In [37]:
# ── Paso 1: Identificar columnas beta ─────────────────────────────
print("⏳ Leyendo header de metilación...")
with gzip.open(CONFIG['met_processed'], 'rt') as f:
    met_header_raw = f.readline().strip()

import csv as csvmod
reader = csvmod.reader([met_header_raw])
met_header = next(reader)

beta_columns = [c for c in met_header[1:] if 'beta' in c.lower()]
print(f"  Columnas beta: {len(beta_columns)}")
print(f"  Index col (met_header[0]): repr='{met_header[0]}'")

# ── Paso 2: Cargar solo betas ─────────────────────────────────────
# index_col=0 se encarga del CpG ID automáticamente,
# usecols con posiciones numéricas: 0 (index) + posiciones de betas
print("⏳ Cargando betas (1-3 min)...")

# Construir set de posiciones numéricas de columnas beta
beta_positions = set()
for i, col in enumerate(met_header):
    if 'beta' in col.lower():
        beta_positions.add(i)

# Posición 0 = index (CpG IDs) + posiciones de betas
usecols_positions = [0] + sorted(beta_positions)
print(f"  Cargando {len(usecols_positions)} columnas de {len(met_header)} totales")

met = pd.read_csv(
    CONFIG['met_processed'],
    compression='gzip',
    index_col=0,
    usecols=usecols_positions,
    low_memory=False,
)
met = met.astype(np.float32)

# Limpiar nombres de columnas: "203784950092_R01C01 beta" → "203784950092_R01C01"
met.columns = [c.replace(' beta', '').strip() for c in met.columns]

print(f"  Shape: {met.shape}")
print(f"  RAM: {met.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print(f"  Rango: [{met.values.min():.4f}, {met.values.max():.4f}]")
print(f"  NaN: {met.isna().sum().sum()}")
print(f"  Index ejemplo: {met.index[:3].tolist()}")
print(f"  Cols ejemplo: {met.columns[:3].tolist()}")

⏳ Leyendo header de metilación...
  Columnas beta: 479
  Index col (met_header[0]): repr=''
⏳ Cargando betas (1-3 min)...
  Cargando 480 columnas de 959 totales
  Shape: (865817, 479)
  RAM: 1.7 GB
  Rango: [0.0000, 0.9984]
  NaN: 0
  Index ejemplo: ['cg14817997', 'cg26928153', 'cg16269199']
  Cols ejemplo: ['203784950092_R01C01', '203784950092_R02C01', '203784950092_R03C01']


### 8b. Seleccionar top CpGs por varianza

In [38]:
# ── Varianza por CpG ──────────────────────────────────────────────
print("⏳ Calculando varianza por CpG...")
cpg_var = met.var(axis=1).sort_values(ascending=False)
print(f"  CpGs con var > 0: {(cpg_var > 0).sum()} / {len(cpg_var)}")
print(f"  Top 5 CpGs: {cpg_var.head(5).to_dict()}")

# ── Seleccionar top N ────────────────────────────────────────────
selected_cpgs = cpg_var.head(CONFIG['n_cpgs_var']).index.tolist()
print(f"  Seleccionados: {len(selected_cpgs)} CpGs")

# ── Extraer submatriz y transponer (CpGs × muestras → muestras × CpGs) ──
met_selected = met.loc[selected_cpgs].T  # → muestras × CpGs
met_selected.index.name = 'met_col'

# Renombrar columnas de sentrix_basename a match limpio
# met_selected index = "203784950092_R01C01 beta" → limpiar
met_selected.index = [idx.replace(' beta', '').strip() for idx in met_selected.index]

print(f"  Submatriz met: {met_selected.shape} (muestras × CpGs)")
print(f"  RAM submatriz: {met_selected.memory_usage(deep=True).sum() / 1e6:.0f} MB")

# Liberar matriz completa
del met
import gc; gc.collect()
print("  🧹 Matriz completa liberada de RAM")

⏳ Calculando varianza por CpG...
  CpGs con var > 0: 865817 / 865817
  Top 5 CpGs: {'cg11796481': 0.22365260124206543, 'cg02216016': 0.2227666974067688, 'cg00004073': 0.2140696793794632, 'cg25099095': 0.20900733768939972, 'cg12657416': 0.20718395709991455}
  Seleccionados: 10000 CpGs
  Submatriz met: (479, 10000) (muestras × CpGs)
  RAM submatriz: 19 MB
  🧹 Matriz completa liberada de RAM


## 9. Hallmark targets — scores derivados de RNA y CSV

Estos proxies supervisarán los heads de hallmarks sobre z fusionado.
Se computan para las muestras que tienen la modalidad correspondiente.

In [39]:
# ── Targets basados en RNA (z-score de genes marcadores) ──────────
def compute_rna_hallmark_score(rna_matrix, gene_list, score_name):
    """Promedio de z-scores de los genes marcadores presentes."""
    genes_present = [g for g in gene_list if g in rna_matrix.columns]
    if not genes_present:
        return pd.Series(np.nan, index=rna_matrix.index, name=score_name)
    subset = rna_matrix[genes_present]
    zscores = (subset - subset.mean()) / subset.std()
    return zscores.mean(axis=1).rename(score_name)

# Calcular scores para muestras con RNA
rna_scores = pd.DataFrame(index=rna_selected.index)
rna_scores['senescence_score'] = compute_rna_hallmark_score(
    rna_selected, HALLMARK_GENES['senescence'], 'senescence_score'
)
rna_scores['sasp_score'] = compute_rna_hallmark_score(
    rna_selected, HALLMARK_GENES['sasp'], 'sasp_score'
)
rna_scores['mito_rna_score'] = compute_rna_hallmark_score(
    rna_selected, HALLMARK_GENES['mitochondrial'], 'mito_rna_score'
)

# Agregar la columna de RNA que mapea a sample_id
#rna_scores['rna_col'] = rna_scores.index  # "Sample_N"

print("Hallmark scores RNA (para muestras con RNA):")
for col in ['senescence_score', 'sasp_score', 'mito_rna_score']:
    vals = rna_scores[col].dropna()
    print(f"  {col}: n={len(vals)}, rango=[{vals.min():.3f}, {vals.max():.3f}]")

# ── Targets escalares (ya en master desde CSV central) ───────────
print(f"\nTargets escalares del CSV central:")
print(f"  telomere_length: {master['telomere_length'].notna().sum()} valores")
print(f"  mtdna_cn: {master['mtdna_cn'].notna().sum()} valores")
for c in clock_cols:
    n = master[c].notna().sum()
    if n > 0:
        print(f"  {c}: {n} valores")

Hallmark scores RNA (para muestras con RNA):
  senescence_score: n=345, rango=[-1.851, 1.207]
  sasp_score: n=345, rango=[-0.886, 1.932]
  mito_rna_score: n=345, rango=[-1.252, 1.882]

Targets escalares del CSV central:
  telomere_length: 496 valores
  mtdna_cn: 493 valores
  clock_horvath1: 494 valores
  clock_horvath2: 494 valores
  clock_hannum: 494 valores
  clock_phenoage: 494 valores
  clock_grimage: 494 valores
  clock_dnamtl: 494 valores
  clock_dnamsen: 494 valores
  clock_skin: 60 valores
  clock_dunedin: 496 valores


## 10. Integrar hallmark scores al master

In [40]:
# ...existing code...
# Mapear RNA scores al master via rna_matrix_col
rna_scores_for_join = rna_scores.reset_index().rename(columns={'rna_col': 'rna_matrix_col'})

score_cols = ['senescence_score', 'sasp_score', 'mito_rna_score', 'rna_matrix_col']
rna_scores_for_join = rna_scores_for_join[score_cols]

n_before = len(master)
master = master.merge(rna_scores_for_join, on='rna_matrix_col', how='left')
assert len(master) == n_before, f"Join expandió filas: {n_before} → {len(master)}"

print("Scores integrados al master:")
print(f"  senescence_score: {master['senescence_score'].notna().sum()} valores")
print(f"  sasp_score: {master['sasp_score'].notna().sum()} valores")
print(f"  mito_rna_score: {master['mito_rna_score'].notna().sum()} valores")
# ...existing code...

Scores integrados al master:
  senescence_score: 345 valores
  sasp_score: 345 valores
  mito_rna_score: 345 valores


## 11. Subconjuntos MVP-2

- **PRETRAIN**: cualquier ómica + PDL (incluye treatments y SURF1 → máximos datos)
- **FINETUNE**: Control + Normal + PDL + alguna ómica (sin SURF1, sin treatments)
- **EVAL**: finetune con 2+ modalidades (para evaluar fusión)

In [41]:
has_any_omic = master['has_rna'] | master['has_met']
has_pdl = master['pdl'].notna()

# ── PRETRAIN: máximos datos ──────────────────────────────────────
mask_pretrain = has_any_omic & has_pdl

# ── FINETUNE: Control + Normal ───────────────────────────────────
is_control = master['treatment_clean'].str.lower().str.contains('control', na=False)
is_healthy = master['clinical_condition'].str.lower().str.contains('normal', na=False)
mask_finetune = mask_pretrain & is_control & is_healthy

# ── EVAL: finetune + 2+ modalidades ─────────────────────────────
mask_eval = mask_finetune & (master['n_modalities'] >= 2)

print("SUBCONJUNTOS MVP-2:")
for name, mask in [('PRETRAIN', mask_pretrain), ('FINETUNE', mask_finetune), ('EVAL', mask_eval)]:
    sub = master[mask]
    print(f"\n  {name}: {mask.sum()} muestras")
    print(f"    RNA={sub['has_rna'].sum()} | Met={sub['has_met'].sum()} | "
          f"Img={sub['has_img'].sum()}")
    print(f"    Cell lines: {sorted(sub['cell_line'].dropna().unique())}")
    if sub['fold'].notna().any():
        for f in sorted(sub['fold'].dropna().unique()):
            nf = (sub['fold'] == f).sum()
            print(f"    Fold {int(f)}: {nf}")

SUBCONJUNTOS MVP-2:

  PRETRAIN: 715 muestras
    RNA=345 | Met=479 | Img=370
    Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2', 'hFB6', 'hFB7', 'hFB8']
    Fold 0: 219
    Fold 1: 335
    Fold 2: 155

  FINETUNE: 161 muestras
    RNA=86 | Met=118 | Img=105
    Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
    Fold 0: 54
    Fold 1: 78
    Fold 2: 29

  EVAL: 146 muestras
    RNA=71 | Met=118 | Img=105
    Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
    Fold 0: 51
    Fold 1: 66
    Fold 2: 29


## 12. Validaciones y contrato de integridad

In [46]:
errors = []

# V1: sample_id único
if not master['sample_id'].is_unique:
    errors.append("sample_id NO es único")

# V2: PDL_norm en [0,1]
pdl_valid = master['pdl_norm'].dropna()
if len(pdl_valid) > 0 and (pdl_valid.min() < -0.001 or pdl_valid.max() > 1.001):
    errors.append(f"PDL_norm fuera de [0,1]: [{pdl_valid.min():.4f}, {pdl_valid.max():.4f}]")

# V3: rna_matrix_col apunta a columnas reales
rna_cols_in_master = set(master['rna_matrix_col'].dropna())
rna_cols_in_matrix = set(rna_selected.index) if hasattr(rna_selected, 'index') else set()
rna_orphan = rna_cols_in_master - rna_cols_in_matrix
if rna_orphan:
    errors.append(f"{len(rna_orphan)} rna_matrix_col no existen en la matriz RNA")

# V4: met_matrix_col apunta a columnas reales
met_cols_in_master = set(master['met_matrix_col'].dropna())
met_cols_in_matrix = set(met_selected.index) if hasattr(met_selected, 'index') else set()
met_orphan = met_cols_in_master - met_cols_in_matrix
if met_orphan:
    errors.append(f"{len(met_orphan)} met_matrix_col no existen en la matriz Met")

# V5: Folds completos en finetune
ft = master[mask_finetune]
if ft['fold'].isna().any():
    errors.append(f"{ft['fold'].isna().sum()} muestras finetune sin fold")

# V6: Al menos 10 muestras por fold en finetune
for f in [0, 1, 2]:
    nf = (ft['fold'] == f).sum()
    if nf < 10:
        errors.append(f"Fold {f} tiene solo {nf} muestras en finetune (mínimo 10)")

# V7: Dimensión latente registrada
print(f"Contrato de interfaz: z ∈ R^{CONFIG['latent_dim']}")
print(f"  Encoder RNA: {len(selected_genes)} genes → z_rna ∈ R^{CONFIG['latent_dim']}")
print(f"  Encoder Met: {len(selected_cpgs)} CpGs → z_met ∈ R^{CONFIG['latent_dim']}")
print(f"  Encoder Bio: escalares → z_bio ∈ R^{CONFIG['latent_dim']}")
print(f"  Encoder Img: 224×224×3 → z_img ∈ R^{CONFIG['latent_dim']} (MVP-1 A2)")
print(f"  Fusión: concat+mask → z ∈ R^{CONFIG['latent_dim']}")

if errors:
    print(f"\n❌ ERRORES ({len(errors)}):")
    for e in errors:
        print(f"  • {e}")
else:
    print(f"\n✅ Todas las validaciones pasaron")

Contrato de interfaz: z ∈ R^256
  Encoder RNA: 2027 genes → z_rna ∈ R^256
  Encoder Met: 10000 CpGs → z_met ∈ R^256
  Encoder Bio: escalares → z_bio ∈ R^256
  Encoder Img: 224×224×3 → z_img ∈ R^256 (MVP-1 A2)
  Fusión: concat+mask → z ∈ R^256

✅ Todas las validaciones pasaron


## 13. Export versionado

In [43]:
output_dir = CONFIG['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)

# ── 13a: Master manifest ─────────────────────────────────────────
master_path = output_dir / f'manifest_mvp2_master_{TIMESTAMP}.parquet'
master_csv_path = output_dir / f'manifest_mvp2_master_{TIMESTAMP}.csv'
try:
    master.to_parquet(master_path, index=False)
    print(f"✅ Master (parquet): {master_path.name} ({master.shape})")
except Exception:
    print("⚠️ pyarrow no disponible, exportando solo CSV")
master.to_csv(master_csv_path, index=False)
print(f"✅ Master (CSV): {master_csv_path.name}")

# ── 13b: Submatrices ómicas ──────────────────────────────────────
rna_path = output_dir / f'mvp2_rna_selected_{TIMESTAMP}.parquet'
try:
    rna_selected.to_parquet(rna_path)
    print(f"✅ RNA submatriz: {rna_path.name} ({rna_selected.shape})")
except Exception:
    rna_csv = output_dir / f'mvp2_rna_selected_{TIMESTAMP}.csv.gz'
    rna_selected.to_csv(rna_csv, compression='gzip')
    print(f"✅ RNA submatriz (CSV): {rna_csv.name}")

met_path = output_dir / f'mvp2_met_selected_{TIMESTAMP}.parquet'
try:
    met_selected.to_parquet(met_path)
    print(f"✅ Met submatriz: {met_path.name} ({met_selected.shape})")
except Exception:
    met_csv = output_dir / f'mvp2_met_selected_{TIMESTAMP}.csv.gz'
    met_selected.to_csv(met_csv, compression='gzip')
    print(f"✅ Met submatriz (CSV): {met_csv.name}")

# ── 13c: Lista de genes y CpGs seleccionados ─────────────────────
genes_path = output_dir / f'mvp2_selected_genes_{TIMESTAMP}.json'
genes_info = {
    'n_hvgs': CONFIG['n_hvgs'],
    'n_total': len(selected_genes),
    'genes': selected_genes,
    'hallmark_categories': {k: v for k, v in HALLMARK_GENES.items()},
    'hallmark_found': sorted(hallmark_found),
    'hallmark_missing': sorted(hallmark_missing),
}
with open(genes_path, 'w') as f:
    json.dump(genes_info, f, indent=2)
print(f"✅ Genes: {genes_path.name}")

cpgs_path = output_dir / f'mvp2_selected_cpgs_{TIMESTAMP}.txt'
with open(cpgs_path, 'w') as f:
    f.write('\n'.join(selected_cpgs))
print(f"✅ CpGs: {cpgs_path.name}")

# ── 13d: Subconjuntos ────────────────────────────────────────────
for name, mask in [('pretrain', mask_pretrain), ('finetune', mask_finetune), ('eval', mask_eval)]:
    sub_path = output_dir / f'manifest_mvp2_{name}_{TIMESTAMP}.csv'
    master[mask].to_csv(sub_path, index=False)
    print(f"✅ {name}: {sub_path.name} ({mask.sum()} filas)")

# ── 13e: Metadata JSON ───────────────────────────────────────────
meta = {
    'timestamp': TIMESTAMP,
    'n_samples': len(master),
    'n_rna': int(master['has_rna'].sum()),
    'n_met': int(master['has_met'].sum()),
    'n_img': int(master['has_img'].sum()),
    'n_rna_met': int((master['has_rna'] & master['has_met']).sum()),
    'n_rna_met_img': int((master['has_rna'] & master['has_met'] & master['has_img']).sum()),
    'n_pretrain': int(mask_pretrain.sum()),
    'n_finetune': int(mask_finetune.sum()),
    'n_eval': int(mask_eval.sum()),
    'n_genes_selected': len(selected_genes),
    'n_cpgs_selected': len(selected_cpgs),
    'latent_dim': CONFIG['latent_dim'],
    'rna_processed_shape': list(rna_selected.shape),
    'met_selected_shape': list(met_selected.shape),
}
meta_path = output_dir / f'manifest_mvp2_meta_{TIMESTAMP}.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"✅ Metadata: {meta_path.name}")

# ── 13f: MD5 ─────────────────────────────────────────────────────
print(f"\nMD5:")
for p in sorted(output_dir.glob(f'*{TIMESTAMP}*')):
    md5 = hashlib.md5(p.read_bytes()).hexdigest()
    print(f"  {p.name}: {md5}")

⚠️ pyarrow no disponible, exportando solo CSV
✅ Master (CSV): manifest_mvp2_master_20260328_143235.csv
✅ RNA submatriz (CSV): mvp2_rna_selected_20260328_143235.csv.gz
✅ Met submatriz (CSV): mvp2_met_selected_20260328_143235.csv.gz
✅ Genes: mvp2_selected_genes_20260328_143235.json
✅ CpGs: mvp2_selected_cpgs_20260328_143235.txt
✅ pretrain: manifest_mvp2_pretrain_20260328_143235.csv (715 filas)
✅ finetune: manifest_mvp2_finetune_20260328_143235.csv (161 filas)
✅ eval: manifest_mvp2_eval_20260328_143235.csv (146 filas)
✅ Metadata: manifest_mvp2_meta_20260328_143235.json

MD5:
  manifest_mvp2_eval_20260328_143235.csv: 0dae1132e51bf8086301edaee876596d
  manifest_mvp2_finetune_20260328_143235.csv: 6812dfc89fc156ef1ab742e6fb63f2fc
  manifest_mvp2_master_20260328_143235.csv: f6f76bb229561936c3f04e251c8e4a19
  manifest_mvp2_meta_20260328_143235.json: 613991d4ac8695d7b5973d824795cf40
  manifest_mvp2_pretrain_20260328_143235.csv: 32c4f12ed6b98266b1e0a9e94c10513a
  mvp2_met_selected_20260328_143235

## 14. Resumen

In [44]:
print("=" * 70)
print("MANIFEST MVP-2 — RESUMEN")
print("=" * 70)
print(f"""
  TABLA MAESTRA: {master.shape[0]} muestras × {master.shape[1]} columnas
  
  MODALIDADES:
    RNA:   {master['has_rna'].sum()} muestras → {len(selected_genes)} genes seleccionados
    Met:   {master['has_met'].sum()} muestras → {len(selected_cpgs)} CpGs seleccionados
    Img:   {master['has_img'].sum()} muestras → z_img de MVP-1 A2
    Telo:  {master['has_telomere'].sum()} muestras
    mtDNA: {master['has_mtdna'].sum()} muestras
    
  PAIRING:
    RNA ∩ Met:       {(master['has_rna'] & master['has_met']).sum()}
    RNA ∩ Met ∩ Img: {(master['has_rna'] & master['has_met'] & master['has_img']).sum()}
    
  SUBCONJUNTOS:
    PRETRAIN: {mask_pretrain.sum()} (cualquier ómica + PDL, todos treatments)
    FINETUNE: {mask_finetune.sum()} (Control + Normal + PDL)
    EVAL:     {mask_eval.sum()} (finetune + 2+ modalidades)
    
  HALLMARK TARGETS:
    senescence_score: {master['senescence_score'].notna().sum()} muestras
    sasp_score:       {master['sasp_score'].notna().sum()} muestras
    mito_rna_score:   {master['mito_rna_score'].notna().sum()} muestras
    telomere_length:  {master['telomere_length'].notna().sum()} muestras
    mtdna_cn:         {master['mtdna_cn'].notna().sum()} muestras
    relojes:          {master['has_clocks'].sum()} muestras
    
  CONTRATO ENCODERS (dim={CONFIG['latent_dim']}):
    E_rna({len(selected_genes)}) → z_rna ∈ R^{CONFIG['latent_dim']}
    E_met({len(selected_cpgs)}) → z_met ∈ R^{CONFIG['latent_dim']}
    E_bio(escalares) → z_bio ∈ R^{CONFIG['latent_dim']}
    E_img(224×224) → z_img ∈ R^{CONFIG['latent_dim']}  [MVP-1 A2, congelado]
    Fusión concat+mask → z → Hallmarks h + ICD [0,1] + PDL_hat
    
  SIGUIENTE PASO:
    1. Baseline ómico: Elastic Net / XGBoost sobre HVGs → PDL
    2. Encoder RNA: MLP sobre {len(selected_genes)} genes → z_rna (256)
    3. Encoder Met: PCA + MLP sobre {len(selected_cpgs)} CpGs → z_met (256)
    4. Encoder Bio: MLP sobre escalares → z_bio (256)
    5. Verificar alineación cross-modal (correlación parcial)
    6. Fusión concat+mask con drop-modality training
""")
print("=" * 70)

MANIFEST MVP-2 — RESUMEN

  TABLA MAESTRA: 1919 muestras × 50 columnas

  MODALIDADES:
    RNA:   345 muestras → 2027 genes seleccionados
    Met:   479 muestras → 10000 CpGs seleccionados
    Img:   973 muestras → z_img de MVP-1 A2
    Telo:  496 muestras
    mtDNA: 493 muestras

  PAIRING:
    RNA ∩ Met:       109
    RNA ∩ Met ∩ Img: 32

  SUBCONJUNTOS:
    PRETRAIN: 715 (cualquier ómica + PDL, todos treatments)
    FINETUNE: 161 (Control + Normal + PDL)
    EVAL:     146 (finetune + 2+ modalidades)

  HALLMARK TARGETS:
    senescence_score: 345 muestras
    sasp_score:       345 muestras
    mito_rna_score:   345 muestras
    telomere_length:  496 muestras
    mtdna_cn:         493 muestras
    relojes:          496 muestras

  CONTRATO ENCODERS (dim=256):
    E_rna(2027) → z_rna ∈ R^256
    E_met(10000) → z_met ∈ R^256
    E_bio(escalares) → z_bio ∈ R^256
    E_img(224×224) → z_img ∈ R^256  [MVP-1 A2, congelado]
    Fusión concat+mask → z → Hallmarks h + ICD [0,1] + PDL_hat

  SIG